### Requirement
Run `sync.ipynb` first

In [7]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [8]:
label_excel = r"D:\Projects\corrosions\tests\labels.xlsx"

In [9]:
df = pd.read_excel(label_excel)

In [10]:
df

,id,year,area,area_code,nomor,segment,pipe_diameter,length,segment_code,cips_protection,normalized_acvg_dcvg_file,normalized_cips_file,normalized_pcm_file
0,0,2025,Jakarta,jakarta-2025,1,Pipa Servis Indonesia Power,16,1.75,pipa-servis-indonesia-power-16,SACP,D:\Projects\corrosions\tests\normalize\acvg_dc...,D:\Projects\corrosions\tests\normalize\cips\ex...,D:\Projects\corrosions\tests\normalize\pcm\exc...
1,1,2025,Jakarta,jakarta-2025,2,RE Martadinata - Jl. Industri Salim Ivomas 2,16,1.70,re-martadinata-jl-industri-salim-ivomas-2-16,SACP,D:\Projects\corrosions\tests\normalize\acvg_dc...,D:\Projects\corrosions\tests\normalize\cips\ex...,D:\Projects\corrosions\tests\normalize\pcm\exc...
2,2,2025,Jakarta,jakarta-2025,3,Jl. Ps. Minggu/ SPBG - Perumahan Koperasi/Jl. ...,10,3.67,jl-ps-minggu-spbg-perumahan-koperasi-jl-g-subr...,SACP,D:\Projects\corrosions\tests\normalize\acvg_dc...,D:\Projects\corrosions\tests\normalize\cips\ex...,D:\Projects\corrosions\tests\normalize\pcm\exc...
3,3,2025,Jakarta,jakarta-2025,4,outlet MRS Pondok Ungu 1 Reducer Pipa 8'' - Te...,10,0.84,outlet-mrs-pondok-ungu-1-reducer-pipa-8-tee-va...,SACP,D:\Projects\corrosions\tests\normalize\acvg_dc...,D:\Projects\corrosions\tests\normalize\cips\ex...,D:\Projects\corrosions\tests\normalize\pcm\exc...
4,4,2025,Jakarta,jakarta-2025,5,Parang Tritis - Ancol,10,1.51,parang-tritis-ancol-10,SACP,D:\Projects\corrosions\tests\normalize\acvg_dc...,D:\Projects\corrosions\tests\normalize\cips\ex...,D:\Projects\corrosions\tests\normalize\pcm\exc...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,90,2024,Cirebon,cirebon-2024,NaN,STD Garawangi - Sungai Cipetir Selatan - BV Ci...,6,5.32,std-garawangi-sungai-cipetir-selatan-bv-cilump...,SACP,D:\Projects\corrosions\tests\normalize\acvg_dc...,D:\Projects\corrosions\tests\normalize\cips\ex...,D:\Projects\corrosions\tests\normalize\pcm\exc...
91,91,2024,Cilegon,cilegon-2024,NaN,Cilegon - Merak (SV 06 Grogol - SV 07),16,4.94,cilegon-merak-sv-06-grogol-sv-07-16,ICCP,NaN,D:\Projects\corrosions\tests\normalize\cips\ex...,D:\Projects\corrosions\tests\normalize\pcm\exc...
92,92,2024,Cilegon,cilegon-2024,NaN,Bojonegara - Suralaya (SV 01 - SV 04),16,16.53,bojonegara-suralaya-sv-01-sv-04-16,ICCP,NaN,D:\Projects\corrosions\tests\normalize\cips\ex...,D:\Projects\corrosions\tests\normalize\pcm\exc...
93,93,2024,Cilegon,cilegon-2024,NaN,Cilegon - Anyer (SV 05 - SV 14),16,13.30,cilegon-anyer-sv-05-sv-14-16,ICCP,NaN,D:\Projects\corrosions\tests\normalize\cips\ex...,D:\Projects\corrosions\tests\normalize\pcm\exc...


In [11]:
def get_province(area):
    if area == 'Tangerang' or area == 'Cilegon':
        return '36'
    if area == 'Jakarta':
        return '31'
    return '32'

In [12]:
results = []

for index in df.index:
    row = df.iloc[index]
    protected = 0.0
    unprotected = 0.0
    medium_to_poor = 0.0
    medium_to_high = 0.0

    df_cips = pd.read_excel(row['normalized_cips_file'])
    df_pcm = pd.read_excel(row['normalized_pcm_file'])

    acvg_file = None if row['normalized_acvg_dcvg_file'] is np.nan else row['normalized_acvg_dcvg_file']
    df_acvg = pd.read_excel(acvg_file) if acvg_file is not None else None
    total_titik_anomali = len(df_acvg) if acvg_file is not None else 0

    df2 = df_cips['condition'].value_counts(normalize=True) * 100.0

    df3 = df_pcm[df_pcm['Current Loss Rate'] < 200]
    df3 = df3['Condition'].value_counts(normalize=True) * 100.0

    if 'PROTECTED' in df2.index:
        protected = df2['PROTECTED']

    if 'OVER PROTECTED' in df2.index:
        protected = df2['OVER PROTECTED'] + protected

    if 'UNPROTECTED' in df2.index:
        unprotected = df2['UNPROTECTED']

    if 'Medium to High' in df3.index:
        medium_to_high = df3['Medium to High']

    if 'Medium to Poor' in df3.index:
        medium_to_poor = df3['Medium to Poor']

    _result = {
        'Year': row['year'],
        'Area': row['area'],
        'Area Code': row['area_code'],
        'Jalur': row['segment'],
        'Segment Code': row['segment_code'],
        'Diameter (inch)': row['pipe_diameter'],
        'CIPS Protection': row['cips_protection'],
        'Panjang (km)': row['length'],
        'Protected': round(protected, 2),
        'Unprotected': round(unprotected, 2),
        'Medium to Poor': round(medium_to_poor, 2),
        'Medium to High': round(medium_to_high, 2),
        'Total Anomali': total_titik_anomali,
    }

    results.append(_result)


In [13]:
cips = pd.DataFrame(results)

In [14]:
cips

,Year,Area,Area Code,Jalur,Segment Code,Diameter (inch),CIPS Protection,Panjang (km),Protected,Unprotected,Medium to Poor,Medium to High,Total Anomali
0,2025,Jakarta,jakarta-2025,Pipa Servis Indonesia Power,pipa-servis-indonesia-power-16,16,SACP,1.75,100.00,0.00,65.57,34.43,3
1,2025,Jakarta,jakarta-2025,RE Martadinata - Jl. Industri Salim Ivomas 2,re-martadinata-jl-industri-salim-ivomas-2-16,16,SACP,1.70,100.00,0.00,49.12,50.88,2
2,2025,Jakarta,jakarta-2025,Jl. Ps. Minggu/ SPBG - Perumahan Koperasi/Jl. ...,jl-ps-minggu-spbg-perumahan-koperasi-jl-g-subr...,10,SACP,3.67,0.44,99.56,57.89,42.11,2
3,2025,Jakarta,jakarta-2025,outlet MRS Pondok Ungu 1 Reducer Pipa 8'' - Te...,outlet-mrs-pondok-ungu-1-reducer-pipa-8-tee-va...,10,SACP,0.84,100.00,0.00,43.18,56.82,2
4,2025,Jakarta,jakarta-2025,Parang Tritis - Ancol,parang-tritis-ancol-10,10,SACP,1.51,32.46,67.54,32.76,67.24,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,2024,Cirebon,cirebon-2024,STD Garawangi - Sungai Cipetir Selatan - BV Ci...,std-garawangi-sungai-cipetir-selatan-bv-cilump...,6,SACP,5.32,100.00,0.00,27.79,72.21,3
91,2024,Cilegon,cilegon-2024,Cilegon - Merak (SV 06 Grogol - SV 07),cilegon-merak-sv-06-grogol-sv-07-16,16,ICCP,4.94,100.00,0.00,35.64,64.36,0
92,2024,Cilegon,cilegon-2024,Bojonegara - Suralaya (SV 01 - SV 04),bojonegara-suralaya-sv-01-sv-04-16,16,ICCP,16.53,85.02,14.98,40.02,59.98,0
93,2024,Cilegon,cilegon-2024,Cilegon - Anyer (SV 05 - SV 14),cilegon-anyer-sv-05-sv-14-16,16,ICCP,13.30,71.33,28.67,33.85,66.15,0


In [15]:
cips.to_excel('results_segment.xlsx', index=False)

In [16]:
grouped_df = cips.groupby(['Year', 'Area', 'Segment Code'])

In [17]:
sum_df = grouped_df.sum()

In [18]:
# sum_df.drop(columns='Diameter (inch)', inplace=True)

In [19]:
sum_df

Area Code  \
Year Area      Segment Code                                                         
2024 Bekasi    rawa-maju-tegal-gede-offtake-deltamas-24               bekasi-2024   
               unisma-kalimalang-10                                   bekasi-2024   
               unisma-pd-ungu-10                                      bekasi-2024   
               unisma-pd-ungu-16                                      bekasi-2024   
     Bogor     jonggol-cimanggis-16                                    bogor-2024   
...                                                                           ...   
2025 Tangerang pertigaan-rs-asobirin-depan-pt-indorama-venture...  tangerang-2025   
               pipa-servis-pt-alam-cendana-4                       tangerang-2025   
               pipa-servis-pt-lucky-indah-keramik-4                tangerang-2025   
               pipa-servis-pt-pelangi-indah-canindo-4              tangerang-2025   
               pipa-servis-pt-pratama-abadi-industri-6             tangerang-2025   

                                                                                                               Jalur  \
Year Area      Segment Code                                                                                            
2024 Bekasi    rawa-maju-tegal-gede-offtake-deltamas-24                    Rawa Maju - Tegal Gede - Offtake Deltamas   
               unisma-kalimalang-10                                                              Unisma - Kalimalang   
               unisma-pd-ungu-10                                                                    Unisma - Pd Ungu   
               unisma-pd-ungu-16                                                                    Unisma - Pd Ungu   
     Bogor     jonggol-cimanggis-16                                                              Jonggol - Cimanggis   
...                                                                                                              ...   
2025 Tangerang pertigaan-rs-asobirin-depan-pt-indorama-venture...  Pertigaan RS Asobirin - Depan PT Indorama Vent...   
               pipa-servis-pt-alam-cendana-4                                             Pipa Servis PT Alam Cendana   
               pipa-servis-pt-lucky-indah-keramik-4                               Pipa Servis PT Lucky Indah Keramik   
               pipa-servis-pt-pelangi-indah-canindo-4                           Pipa Servis PT Pelangi Indah Canindo   
               pipa-servis-pt-pratama-abadi-industri-6                         Pipa Servis PT Pratama Abadi Industri   

                                                                   Diameter (inch)  \
Year Area      Segment Code                                                          
2024 Bekasi    rawa-maju-tegal-gede-offtake-deltamas-24                         24   
               unisma-kalimalang-10                                             10   
               unisma-pd-ungu-10                                                10   
               unisma-pd-ungu-16                                                16   
     Bogor     jonggol-cimanggis-16                                             16   
...                                                                            ...   
2025 Tangerang pertigaan-rs-asobirin-depan-pt-indorama-venture...               10   
               pipa-servis-pt-alam-cendana-4                                     4   
               pipa-servis-pt-lucky-indah-keramik-4                              4   
               pipa-servis-pt-pelangi-indah-canindo-4                            4   
               pipa-servis-pt-pratama-abadi-industri-6                           6   

                                                                  CIPS Protection  \
Year Area      Segment Code                                                         
2024 Bekasi    rawa-maju-tegal-gede-offtake-deltamas-24                      ICCP   
               unisma-kalimalan

In [20]:
## Cek
sum_df.to_excel('sum_df.xlsx')

In [42]:
results_area = []

for year, area, segment_code in sum_df.index:
    total_condition = sum_df.loc[year, area, segment_code]['Protected'] + sum_df.loc[year, area, segment_code]['Unprotected']
    total_protected = sum_df.loc[year, area, segment_code]['Medium to Poor'] + sum_df.loc[year, area, segment_code]['Medium to High']
    total_anomaly = sum_df.loc[year, area, segment_code]['Total Anomali']

    print(year,area, sum_df.loc[year, area, segment_code]['Protected']/total_condition, total_protected, total_anomaly)

    _result_area = {
        'Year': year,
        'Area': area,
        'Total Panjang (km)': round(sum_df.loc[year, area, segment_code]['Panjang (km)'], 2),
        'Protected': round(sum_df.loc[year, area, segment_code]['Protected']/total_condition*100, 2),
        'Unprotected': round(sum_df.loc[year, area, segment_code]['Unprotected']/total_condition*100, 2),
        'Medium to Poor': round(sum_df.loc[year, area, segment_code]['Medium to Poor']/total_protected*100, 2),
        'Medium to High': round(sum_df.loc[year, area, segment_code]['Medium to High']/total_protected*100, 2),
        'Total Anomali': total_anomaly,
    }

    results_area.append(_result_area)

2024 Bekasi 0.7182 100.0 3
2024 Bekasi 0.1819 100.0 0
2024 Bekasi 0.009000000000000001 100.0 7
2024 Bekasi 0.6514 100.0 2
2024 Bogor 0.9351999999999999 100.0 8
2024 Cilegon 0.8502 100.0 0
2024 Cilegon 0.7132999999999999 100.0 0
2024 Cilegon 1.0 100.0 0
2024 Cilegon 0.0 100.0 0
2024 Cirebon 0.0 100.0 4
2024 Cirebon 1.0 100.0 3
2024 Cirebon 0.1186 100.0 2
2024 Cirebon 0.0 100.0 2
2024 Cirebon 0.46 100.0 2
2024 Jakarta 0.8945000000000001 100.0 0
2024 Jakarta 0.0875 100.0 9
2024 Jakarta 0.3337 100.0 23
2024 Karawang 0.9084 100.0 3
2024 Tangerang 0.9664 100.0 19
2024 Tangerang 0.6286999999999999 100.0 1
2024 Tangerang 0.9253 100.0 2
2024 Tangerang 0.5772999999999999 100.0 9
2025 Bekasi 0.0 100.0 0
2025 Bekasi 0.47240000000000004 100.0 17
2025 Bekasi 0.9351 100.0 7
2025 Bekasi 1.0 100.0 34
2025 Bekasi 0.0253 100.0 0
2025 Bogor 0.8929 100.0 9
2025 Bogor 0.0 100.0 1
2025 Bogor 0.020099999999999996 100.0 5
2025 Bogor 0.8946 100.0 2
2025 Bogor 0.0335 100.0 7
2025 Bogor 0.179 100.0 3
2025 Bogor 0

In [43]:
results_area = pd.DataFrame(results_area)

In [45]:
grouped_year_area = results_area.groupby(['Year', 'Area']).sum()

In [46]:
grouped_year_area

Total Panjang (km)  Protected  Unprotected  Medium to Poor  \
Year Area                                                                    
2024 Bekasi                  35.74     156.05       243.95          140.84   
     Bogor                    9.95      93.52         6.48           53.38   
     Cilegon                 37.89     256.35       143.65          134.31   
     Cirebon                 18.75     157.86       342.14          201.51   
     Jakarta                 24.50     131.57       168.43          125.36   
     Karawang                16.92      90.84         9.16           28.44   
     Tangerang               76.69     309.77        90.23          122.02   
2025 Bekasi                  28.00     243.28       256.72          220.20   
     Bogor                   46.30     390.60       409.40          381.66   
     Cilegon                 20.53     936.55       163.45          503.35   
     Cirebon                 33.17     287.61       212.39          154.11   
     Jakarta                 38.82     740.17       459.83          584.15   
     Karawang                41.00     775.92       124.08          338.25   
     Tangerang               32.19    1632.09       667.91          872.79   

                Medium to High  Total Anomali  
Year Area                                      
2024 Bekasi             259.16             12  
     Bogor               46.62              8  
     Cilegon            265.69              0  
     Cirebon            298.49             13  
     Jakarta            174.64             32  
     Karawang            71.56              3  
     Tangerang          277.98             31  
2025 Bekasi             279.80             58  
     Bogor              418.34             48  
     Cilegon            596.65              2  
     Cirebon            345.89             32  
     Jakarta            615.85             85  
     Karawang           561.75             46  
     Tangerang         1427.21            130

In [48]:
results_year = []

for year, area in grouped_year_area.index:
    total_condition = grouped_year_area.loc[year, area]['Protected'] + grouped_year_area.loc[year, area]['Unprotected']
    total_protected = grouped_year_area.loc[year, area]['Medium to Poor'] + grouped_year_area.loc[year, area]['Medium to High']
    total_anomaly = grouped_year_area.loc[year, area]['Total Anomali']

    _result_year = {
        'Year': year,
        'Area': area,
        'Total Panjang (km)': round(grouped_year_area.loc[year, area]['Total Panjang (km)'], 2),
        'Protected': round(grouped_year_area.loc[year, area]['Protected']/total_condition*100, 2),
        'Unprotected': round(grouped_year_area.loc[year, area]['Unprotected']/total_condition*100, 2),
        'Medium to Poor': round(grouped_year_area.loc[year, area]['Medium to Poor']/total_protected*100, 2),
        'Medium to High': round(grouped_year_area.loc[year, area]['Medium to High']/total_protected*100, 2),
        'Total Anomali': total_anomaly,
    }

    results_year.append(_result_year)

In [49]:
results_year_df = pd.DataFrame(results_year)

In [50]:
results_year_df

,Year,Area,Total Panjang (km),Protected,Unprotected,Medium to Poor,Medium to High,Total Anomali
0,2024,Bekasi,35.74,39.01,60.99,35.21,64.79,12.0
1,2024,Bogor,9.95,93.52,6.48,53.38,46.62,8.0
2,2024,Cilegon,37.89,64.09,35.91,33.58,66.42,0.0
3,2024,Cirebon,18.75,31.57,68.43,40.30,59.70,13.0
4,2024,Jakarta,24.50,43.86,56.14,41.79,58.21,32.0
5,2024,Karawang,16.92,90.84,9.16,28.44,71.56,3.0
6,2024,Tangerang,76.69,77.44,22.56,30.51,69.50,31.0
7,2025,Bekasi,28.00,48.66,51.34,44.04,55.96,58.0
8,2025,Bogor,46.30,48.82,51.18,47.71,52.29,48.0
9,2025,Cilegon,20.53,85.14,14.86,45.76,54.24,2.0


In [51]:
results_year_df.to_excel('results_area.xlsx', index=False)